# Lektion 05 - Agentisk RAG


## Setup

Den här anteckningsboken demonstrerar Agentic RAG (Retrieval-Augmented Generation)-mönstret med Microsoft Agent Framework.

**Förutsättningar:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — din Azure AI Search-tjänständpunkt
- `AZURE_SEARCH_API_KEY` — din Azure AI Search API-nyckel
- Azure OpenAI-distribution konfigurerad via miljövariabler
- Azure CLI autentiserad (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Vad är Agentic RAG?

Traditionell RAG följer en fast pipeline: hämta dokument, sedan generera ett svar. **Agentic RAG** går längre genom att ge agenten autonomi att bestämma **när** och **hur** information ska hämtas.

Med Agentic RAG kan agenten:
- **Bestämma** om hämtning behövs innan en fråga besvaras
- **Välja** vilken datakälla eller verktyg som ska frågas
- **Utvärdera** hämtade resultat och göra uppföljande hämtningar om första försöket är otillräckligt
- **Kombinera** information från flera hämtsteg till ett sammanhängande svar

Detta gör agenten mer flexibel och exakt jämfört med en statisk hämta-sedan-generera-pipeline.


## Skapa ett sökverktyg

I Agentic RAG paketeras externa datakällor som **verktyg** som agenten kan anropa efter behov. Detta låter agenten behandla hämtning som bara en annan åtgärd den kan utföra, snarare än ett obligatoriskt steg.

Nedan definierar vi en resekunskapsbas och exponerar den som ett verktyg som agenten kan använda för att slå upp destinationsinformation.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Skapa RAG-agenten

Nu skapar vi en agent som är instruerad att **alltid hämta information innan den svarar**. Agenten använder verktyget `search_travel_knowledge` för att förankra sina svar i kunskapsbasen istället för att förlita sig på sin egen träningsdata.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iterativ hämtning — Maker-Checker-mönstret

En viktig fördel med Agentic RAG är **iterativ hämtning**. Agenten kan utföra flera sökomgångar för att verifiera, förfina eller utöka sina initiala fynd — liknande en "maker-checker"-arbetsflöde:

1. **Maker-steg**: Agenten hämtar initial information och utarbetar ett svar.
2. **Checker-steg**: Agenten utför ytterligare hämtningar för att kontrollera detaljer eller fylla i luckor.

Nedan blir agenten tillfrågad om en fråga som kräver jämförelse av flera destinationer, vilket får den att söka flera gånger.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Summary

I denna lektion lärde du dig hur man bygger ett **Agentic RAG**-system med Microsoft Agent Framework:

- **Agentic RAG** låter agenter självständigt besluta när de ska hämta information, vilket gör hämtningen dynamisk istället för fast.
- **Verktyg som datakällor**: Externa kunskapsbaser (som Azure AI Search) är inbäddade som verktyg som agenten kan anropa.
- **Iterativ hämtning**: Maker-checker-mönstret gör det möjligt för agenten att utföra flera hämtomgångar — söka, verifiera och förfina — innan ett slutgiltigt svar produceras.

I produktion skulle du ersätta den minnesbaserade `TRAVEL_KNOWLEDGE_BASE` med ett riktigt Azure AI Search-index för att hantera storskalig hämtning av resehandlingar.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:  
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, var vänlig notera att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess originalspråk ska betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår från användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
